# E2 bổ sung: NCM cosine và chia lại ImageNet-R/-A

Kiểm chứng hai giả thuyết trong `Docs/report4.md`:

1. **Mục 3.3:** NCM của nhóm thấp hơn bài có phải vì dùng khoảng cách Euclid? Chạy lại 8 run NCM với **cosine**.
2. **Mục 3.2:** ImageNet-A cao hơn bài có phải vì cách chia train/test? Chia lại ImageNet-R và ImageNet-A với seed 1-4 (`--split-seed`), chạy RanDumb và NCM Euclid, xem số dao động bao nhiêu.

**Chuẩn bị trên Kaggle:**
1. **Add Input → Your Work → Notebooks**, chọn notebook E2 đã **Save Version** (`02_e2_vit`). Ô 4 tự tìm file `data/features/*.pt` trong đó, khỏi trích lại khoảng 1 giờ.
2. **Internet: On**, Secret `WANDB_API_KEY`. Có file đặc trưng thì không cần GPU; nếu thiếu, ô 6 tự trích và khi đó cần GPU T4.

**Thời gian:** khoảng 10 phút nếu đã có file đặc trưng (40 run, mỗi run vài giây, phần lớn là thời gian mở run W&B).

In [ ]:
# Ô 1. Cấu hình
REPO_URL = "https://github.com/24122039Kien/RanDumb-reproduction.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/RanDumb-reproduction"
DATA_DIR = "/kaggle/working/data"
INPUT_DIR = "/kaggle/input"

BACKBONES = ["in21k", "in1k"]
COSINE_DATASETS = ["cifar100", "cub", "imagenet-r", "imagenet-a"]
RESPLIT_DATASETS = ["imagenet-a", "imagenet-r"]
SPLIT_SEEDS = [1, 2, 3, 4]
RESPLIT_METHODS = [("randumb", "euclidean"), ("ncm", "euclidean")]
TAGS = ["lab:reproduction", "E2", "E2-followup"]

# Số của lần chạy E2 chính (Docs/report4.md, tập chia SPLIT_SEED = 0) để so sánh.
MAIN_RUN = {
    ("imagenet-a", "in21k", "randumb"): 49.33, ("imagenet-a", "in21k", "ncm"): 37.27,
    ("imagenet-a", "in1k", "randumb"): 55.00, ("imagenet-a", "in1k", "ncm"): 44.13,
    ("imagenet-r", "in21k", "randumb"): 65.04, ("imagenet-r", "in21k", "ncm"): 50.23,
    ("imagenet-r", "in1k", "randumb"): 66.69, ("imagenet-r", "in1k", "ncm"): 59.30,
    ("cifar100", "in21k", "ncm"): 79.28, ("cifar100", "in1k", "ncm"): 82.68,
    ("cub", "in21k", "ncm"): 84.02, ("cub", "in1k", "ncm"): 84.21,
}

In [ ]:
# Ô 2. Lấy code của nhóm
import os
import subprocess
import sys

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
print("Commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
               check=True)

In [ ]:
# Ô 3. Secrets, biến môi trường, test của repo
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass
os.environ["RANDUMB_DATA_DIR"] = DATA_DIR
os.environ["PYTHONUNBUFFERED"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=REPO_DIR, check=True)

In [ ]:
# Ô 4. Lấy file đặc trưng từ output của notebook E2 (đã Add Input)
from pathlib import Path

features_dir = Path(DATA_DIR) / "features"
features_dir.mkdir(parents=True, exist_ok=True)
found = sorted(Path(INPUT_DIR).rglob("*-norm.pt")) if Path(INPUT_DIR).exists() else []
for src in found:
    dst = features_dir / src.name
    if not dst.exists():
        dst.symlink_to(src)
print(f"Tìm thấy {len(found)} file đặc trưng trong {INPUT_DIR}:")
for path in sorted(features_dir.glob("*.pt")):
    print(" ", path.name)
if not found:
    print("Không có file nào: ô 6 sẽ trích lại (cần GPU, khoảng 1 giờ cho cả 4 dataset).")

In [ ]:
# Ô 5. Hàm chạy script của repo
import json
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", DEVICE)


def run(script, *args):
    """Chạy một script của repo, in log khi lỗi, trả về JSON in ở cuối (nếu có)."""
    cmd = [sys.executable, os.path.join(REPO_DIR, "scripts", script), *args]
    proc = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-1500:])
        print(proc.stderr[-3000:])
        raise RuntimeError(f"{script} lỗi với {args}")
    text = "\n" + proc.stdout
    start = text.rfind("\n{")
    return json.loads(text[start:]) if start != -1 else None


def ensure_features(dataset, backbone):
    """Trích đặc trưng nếu chưa có; script tự bỏ qua khi file đã tồn tại."""
    run("extract_vit_features.py", "--dataset", dataset, "--backbone", backbone, "--device", DEVICE)


tag_args = [arg for tag in TAGS for arg in ("--tag", tag)]

In [ ]:
# Ô 6. Chạy: 8 run NCM cosine, rồi 2 dataset × 2 backbone × 4 seed chia × 2 phương pháp = 32 run
results = []
for dataset in COSINE_DATASETS:
    for backbone in BACKBONES:
        ensure_features(dataset, backbone)
        out = run("run_e2.py", "--dataset", dataset, "--backbone", backbone, "--method", "ncm",
                  "--ncm-metric", "cosine", "--device", DEVICE, *tag_args)
        results.append({"experiment": "ncm-cosine", "dataset": dataset, "backbone": backbone,
                        "method": "ncm-cosine", "split_seed": 0, **out})
        print(f"{dataset}/{backbone} NCM cosine: {out['final_accuracy']:.2f}%")

for dataset in RESPLIT_DATASETS:
    for backbone in BACKBONES:
        ensure_features(dataset, backbone)
        for split_seed in SPLIT_SEEDS:
            for method, metric in RESPLIT_METHODS:
                out = run("run_e2.py", "--dataset", dataset, "--backbone", backbone, "--method", method,
                          "--ncm-metric", metric, "--split-seed", str(split_seed), "--device", DEVICE, *tag_args)
                results.append({"experiment": "resplit", "dataset": dataset, "backbone": backbone,
                                "method": method, "split_seed": split_seed, **out})
                print(f"{dataset}/{backbone}/{method} chia seed {split_seed}: {out['final_accuracy']:.2f}%")

In [ ]:
# Ô 7. Bảng kết quả để dán vào Docs và deck
import pandas as pd

pd.set_option("display.max_colwidth", 200)
df = pd.DataFrame(results)

cos = df[df.experiment == "ncm-cosine"].copy()
cos["euclidean_main_run"] = [MAIN_RUN.get((d, b, "ncm")) for d, b in zip(cos.dataset, cos.backbone)]
cos = cos[["dataset", "backbone", "final_accuracy", "euclidean_main_run", "paper_accuracy",
           "relative_deviation_pct", "wandb_url"]].round(2)
print("1) NCM cosine so với NCM Euclid (lần chạy chính) và số của bài")
display(cos)

rs = df[df.experiment == "resplit"]
summary = (rs.groupby(["dataset", "backbone", "method"])["final_accuracy"]
             .agg(["mean", "std", "min", "max"]).reset_index())
summary["main_run_seed0"] = [MAIN_RUN.get(k) for k in zip(summary.dataset, summary.backbone, summary.method)]
summary["paper_accuracy"] = [rs[(rs.dataset == d) & (rs.backbone == b) & (rs.method == m)].paper_accuracy.iloc[0]
                             for d, b, m in zip(summary.dataset, summary.backbone, summary.method)]
print("2) Chia lại train/test: trung bình, độ lệch chuẩn mẫu, min, max qua", len(SPLIT_SEEDS), "seed chia")
display(summary.round(2))

runs = rs[["dataset", "backbone", "method", "split_seed", "final_accuracy", "n_train", "n_test", "wandb_url"]]
for name, table in [("ncm_cosine", cos), ("resplit_summary", summary.round(2)), ("resplit_runs", runs.round(2))]:
    table.to_csv(f"/kaggle/working/e2_followup_{name}.csv", index=False)
    try:
        print(f"\n### {name}\n" + table.to_markdown(index=False))
    except ImportError:  # thiếu thư viện tabulate
        print(f"\n### {name}\n" + table.to_csv(index=False))

## Cách đọc kết quả

- **NCM cosine:** nếu cosine gần số của bài hơn Euclid (nhất là với IN-21K), giả thuyết 3.3 đúng. Ghi cả hai vào deck, nói rõ bài không nêu khoảng cách.
- **Chia lại:** nếu `max − min` của ImageNet-A cỡ vài điểm phần trăm, cách chia ảnh hưởng mạnh. Nếu cả 4 seed vẫn cao hơn bài 5-7 điểm, nguyên nhân nằm ở chỗ khác (trọng số backbone, tiền xử lý), không phải may rủi của một lần chia.
- Sau khi chạy: **Save Version**, rồi gửi 3 bảng ở ô 7 (file CSV nằm ở `/kaggle/working`).